# GNN Comparison on QM9 — Training Notebook
**Models:** GCN · GAT · GATv2 · SchNet  
**Dataset:** QM9 (Ramakrishnan et al., 2014)  
**Task:** Molecular property regression

We include both GAT and GATv2 because GATv2 incorporates bond-type edge attributes
into its attention scores (via `edge_dim`), while standard GAT computes attention from
node features alone. Comparing them isolates the effect of edge-aware attention.

---
**Instructions:**
1. Mount Google Drive (Cell 1) — logs/checkpoints save there
2. Install dependencies (Cell 2)
3. Clone/upload your project repo (Cell 3)
4. Load data (Cell 4)
5. Train GCN (Cell 5) · GAT (Cell 6) · GATv2 (Cell 7) · SchNet (Cell 8)
6. Evaluate all models on test set (Cell 9)
7. Plot training curves (Cell 10)

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/gnn_qm9'

import os
os.makedirs(f'{DRIVE_ROOT}/outputs/logs',        exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/results',     exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data',                exist_ok=True)
print('Drive mounted. Output root:', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Install Dependencies ─────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = 'cu121' if torch.cuda.is_available() else 'cpu'

!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html
!pip install -q pyyaml tqdm

print('\nInstallation complete.')

In [ ]:
# ── Cell 3: Clone Project Repo ───────────────────────────────────────────────
import os, sys

!git clone https://github.com/amanikonda123/DL-Final-Project.git /content/gnn_qm9

PROJECT_DIR = '/content/gnn_qm9'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Project files:', os.listdir('.'))

In [ ]:
# ── Cell 4: Load Data ─────────────────────────────────────────────────────────
# Shared data loaders — all three models use the same splits and normalization.
# Each model's config overrides feature_mode internally; the loaders are reused.

import torch
import yaml
from data.loader import get_dataloaders

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = f'{DRIVE_ROOT}/data/qm9_raw'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'

print('Device:', DEVICE)

# Load per-model configs from YAML
with open('config/gcn.yaml')    as f: GCN_CFG    = yaml.safe_load(f)
with open('config/gat.yaml')    as f: GAT_CFG    = yaml.safe_load(f)
with open('config/gatv2.yaml')  as f: GATV2_CFG  = yaml.safe_load(f)
with open('config/schnet.yaml') as f: SCHNET_CFG = yaml.safe_load(f)

# Sanity check — inspect one batch using GCN config (topology mode)
train_loader, val_loader, test_loader, normalizer = get_dataloaders(GCN_CFG, root=DATA_ROOT)
batch = next(iter(train_loader))
print('\nBatch fields:    ', list(batch.keys))
print('x shape:         ', batch.x.shape)
print('edge_index shape:', batch.edge_index.shape)
print('y shape:         ', batch.y.shape)
print('pos shape:       ', batch.pos.shape)
print('z shape:         ', batch.z.shape)
print('num_graphs:      ', batch.num_graphs)

In [ ]:
# ── Cell 5: Train GCN ─────────────────────────────────────────────────────────
from train import train_model

gcn_result = train_model(
    GCN_CFG,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gcn',
)
print(f"\nGCN  — Best val MAE: {gcn_result['best_val_mae']:.6f}")

In [ ]:
# ── Cell 6: Train GAT ─────────────────────────────────────────────────────────
from train import train_model

gat_result = train_model(
    GAT_CFG,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gat',
)
print(f"\nGAT  — Best val MAE: {gat_result['best_val_mae']:.6f}")

In [ ]:
# ── Cell 7: Train GATv2 ──────────────────────────────────────────────────────
from train import train_model

gatv2_result = train_model(
    GATV2_CFG,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='gatv2',
)
print(f"\nGATv2 — Best val MAE: {gatv2_result['best_val_mae']:.6f}")

In [ ]:
# ── Cell 8: Train SchNet ──────────────────────────────────────────────────────
from train import train_model

schnet_result = train_model(
    SCHNET_CFG,
    device=str(DEVICE),
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    model_name='schnet',
)
print(f"\nSchNet — Best val MAE: {schnet_result['best_val_mae']:.6f}")

In [ ]:
# ── Cell 9: Evaluate All Models on Test Set ───────────────────────────────────
from evaluate import evaluate_model

results = {}

for model_name, cfg, ckpt in [
    ('gcn',    GCN_CFG,    gcn_result['checkpoint']),
    ('gat',    GAT_CFG,    gat_result['checkpoint']),
    ('gatv2',  GATV2_CFG,  gatv2_result['checkpoint']),
    ('schnet', SCHNET_CFG, schnet_result['checkpoint']),
]:
    print(f'\n── {model_name.upper()} ──')
    results[model_name] = evaluate_model(
        cfg,
        checkpoint_path=ckpt,
        device=str(DEVICE),
        output_dir=OUTPUT_DIR,
        data_root=DATA_ROOT,
        model_name=model_name,
    )

print('\n── Summary ──')
for name, r in results.items():
    print(f"{name.upper():8s}  MAE={r['test_mae']:.6f}  RMSE={r['test_rmse']:.6f}")

In [ ]:
# ── Cell 10: Training Curves ──────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

models = {
    'GCN':    f'{OUTPUT_DIR}/logs/gcn_history.csv',
    'GAT':    f'{OUTPUT_DIR}/logs/gat_history.csv',
    'GATv2':  f'{OUTPUT_DIR}/logs/gatv2_history.csv',
    'SchNet': f'{OUTPUT_DIR}/logs/schnet_history.csv',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, path in models.items():
    h = pd.read_csv(path)
    axes[0].plot(h['epoch'], h['val_loss'], label=name)
    axes[1].plot(h['epoch'], h['val_mae'],  label=name)

axes[0].set(xlabel='Epoch', ylabel='MSE Loss (normalized)', title='Validation Loss')
axes[1].set(xlabel='Epoch', ylabel='MAE (denormalized)',    title='Validation MAE')
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/results/all_models_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR}/results/all_models_training_curves.png")